# Lab Task: File Handling, Exceptions, and OOP in Python

**Duration:** 120 minutes  
**Level:** MS Data Science Students  
**Difficulty:** Medium–High  

---
## Scenario
You are working as a **data engineer** for a small fintech startup. Your tasks involve managing transaction records stored in files, handling unexpected errors during processing, and designing object-oriented classes to model customer accounts.
---

## Part 1: File Handling and Exceptions (45 minutes)

### Task 1.1 – Reading & Handling Missing Files
- A file named `transactions.txt` is expected to contain one transaction per line in the form:
  ```
  deposit,1000
  withdraw,500
  withdraw,0
  ```
- Write a program that:
  1. Attempts to open and read `transactions.txt`.
  2. If the file does not exist, catch the `FileNotFoundError`, create a new empty file with that name, and print a message.


In [ ]:
# First run: file not found, so the except block runs.
# Next runs: file exists, so try block reads it.
try:
    with open('transactions.txt', 'r') as f:
        transactions = f.readlines()

# Runs only the first time when the file isn’t there.
except FileNotFoundError:
    # Make a new empty file if missing
    with open('transactions.txt', 'w') as f:
        pass
    # Shown only on first run.
    print("No existing file found. A fresh file has been generated.")
    # Start with an empty list.
    transactions = []

### Task 1.2 – Processing Transactions
- Extend your program to:
  1. Read transactions line by line.
  2. For each transaction, process deposits and withdrawals.
  3. Handle possible errors:
     - `ValueError` (if data format is wrong)
     - `ZeroDivisionError` (simulate when withdrawal amount is `0`)
     - `PermissionError`
     - Any unexpected exceptions
  4. Always print `Transaction processed` in a `finally` block.


In [2]:
with open('transactions.txt', 'w') as f:
    f.write("deposit, 1000\n")
    f.write("withdraw, 500\n")
    f.write("withdraw, 100\n")
    f.write("withdraw, 0\n")   # tests ZeroDivisionError
    f.write("withdraw, abc\n") # tests ValueError (invalid number)
    f.write("collect, 100\n")  # tests ValueError (invalid action)


try:
    with open('transactions.txt', 'r') as f:
        transactions = f.readlines()
except FileNotFoundError:
    with open('transactions.txt', 'w') as f:
        pass
    print("File not found. Created a new file.")
    transactions = []

print("Transactions loaded:", transactions)

Transactions loaded: ['deposit, 1000\n', 'withdraw, 500\n', 'withdraw, 100\n', 'withdraw, 0\n', 'withdraw, abc\n', 'collect, 100\n']


In [3]:
balance = 0

for line in transactions:    # go through each transaction one by one
    try:
        action, amount = line.strip().split(',')   # split into action and amount
        amount = int(amount)   # turn amount into a number

        if action == 'deposit':      # add money if deposit
            balance += amount
        elif action == 'withdraw':   # subtract money if withdraw
            if amount == 0:          # not allowed to withdraw 0
                _ = 100/0            # force ZeroDivisionError
            balance -= amount
        else:
            raise ValueError("Unknown action")   # any other action is invalid

    except ValueError as ve:         # wrong number or unknown action
        print(f"ValueError: {ve}")
    except ZeroDivisionError:        # trying to withdraw 0
        print("ZeroDivisionError: Cannot withdraw 0.")
    except PermissionError:          # if file access was blocked
        print("PermissionError: Access denied.")
    except Exception as e:           # anything else unexpected
        print(f"Unexpected error: {e}")
    finally:                         # always runs, no matter what
        print("Transaction processed")

print("Final Balance:", balance)     # show balance at the end

Transaction processed
Transaction processed
Transaction processed
ZeroDivisionError: Cannot withdraw 0.
Transaction processed
Invalid amount:  abc
Transaction processed
ValueError: Unknown action
Transaction processed
Final Balance: 400


### Task 1.3 – Writing and Appending
- After processing:
  1. Write the final balance into `balance.txt`.
  2. Append successful transactions into `log.txt` with a timestamp.
  3. Append failed transactions into `error_log.txt`.


In [5]:
from datetime import datetime

balance = 0  # start with 0 balance

for line in transactions:   # go through each transaction
    try:
        action, amount = line.strip().split(',')   # split into action and amount
        amount = int(amount)   # convert amount to number

        if action == 'deposit':
            balance += amount
            # record successful deposit
            with open('log.txt', 'a') as lf:
                lf.write(f"{datetime.now()}: {action} {amount}, Balance = {balance}\n")

        elif action == 'withdraw':
            if amount == 0:
                _ = 100/0   # force error for withdraw 0
            balance -= amount
            # record successful withdrawal
            with open('log.txt', 'a') as lf:
                lf.write(f"{datetime.now()}: {action} {amount}, Balance = {balance}\n")

        else:
            raise ValueError("Unknown action")   # if action is not deposit/withdraw

    except ValueError as ve:
        # log invalid action or bad number
        with open('error_log.txt', 'a') as ef:
            ef.write(f"{datetime.now()}: ValueError - {ve}, Transaction = {line}\n")

    except ZeroDivisionError:
        # log withdraw = 0 error
        with open('error_log.txt', 'a') as ef:
            ef.write(f"{datetime.now()}: ZeroDivisionError - Cannot withdraw 0, Transaction = {line}\n")

    except PermissionError:
        # log file permission issues
        with open('error_log.txt', 'a') as ef:
            ef.write(f"{datetime.now()}: PermissionError - Access denied, Transaction = {line}\n")

    except Exception as e:
        # log any other unexpected error
        with open('error_log.txt', 'a') as ef:
            ef.write(f"{datetime.now()}: Unexpected error - {e}, Transaction = {line}\n")

    finally:
        print("Transaction processed")   # always runs

# save final balance
with open('balance.txt', 'w') as bf:
    bf.write(f"Final Balance: {balance}\n")

# also log final balance in log.txt
with open('log.txt', 'a') as lf:
    lf.write(f"{datetime.now()}: Final Balance = {balance}\n")

# show balance.txt content
with open('balance.txt', 'r') as bf:
    print("\nbalance.txt")
    print(bf.read())

# show log.txt content
with open('log.txt', 'r') as lf:
    print("log.txt")
    print(lf.read())

# show error_log.txt content
with open('error_log.txt', 'r') as ef:
    print("error_log.txt")
    print(ef.read())

Transaction processed
Transaction processed
Transaction processed
Transaction processed
Transaction processed
Transaction processed

balance.txt
Final Balance: 400

log.txt
2025-10-01 14:26:33.642130: deposit 1000, Balance = 1000
2025-10-01 14:26:33.643446: withdraw 500, Balance = 500
2025-10-01 14:26:33.643530: withdraw 100, Balance = 400
2025-10-01 14:26:33.644198: Final Balance = 400

error_log.txt
2025-10-01 14:26:33.643641: ZeroDivisionError - Cannot withdraw 0, Transaction = withdraw, 0

2025-10-01 14:26:33.643710: ValueError - invalid literal for int() with base 10: ' abc', Transaction = withdraw, abc

2025-10-01 14:26:33.643772: ValueError - Unknown action, Transaction = collect, 100




## Part 2: Object-Oriented Programming (75 minutes)

### Task 2.1 – Defining the Class
- Create a class `BankAccount` with attributes, constructor, deposit, withdraw, custom exception handling, and `__str__` method.

In [16]:
# custom error if account has not enough money
class InsufficientFundsError(Exception):
    pass

# BankAccount class
class BankAccount:
    interest_rate = 0.05   # interest rate for all accounts

    # set up account with number, owner, and balance
    def __init__(self, account_number, owner_name, balance=0):
        self.account_number = account_number
        self.owner_name = owner_name
        self.balance = balance

    # put money in the account
    def deposit(self, amount):
        self.balance += amount

    # take money out (error if not enough balance)
    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError("Not enough funds.")
        self.balance -= amount

    # add interest to balance
    def apply_interest(self):
        self.balance += self.balance * BankAccount.interest_rate

    # how account looks when printed
    def __str__(self):
        return f"Account({self.account_number}, Owner: {self.owner_name}, Balance: {self.balance})"

### Task 2.2 – Using the Class
- Create objects, perform deposits, withdrawals, handle exceptions, update attributes, and write account summary into a file.

In [17]:
# open Alice’s account with 500 balance
acc = BankAccount("12345", "Alice", 500)

# add 200 (balance → 700)
acc.deposit(200)

# try to take out 1000 (too much, so it will raise error)
try:
    acc.withdraw(1000)
except InsufficientFundsError as e:
    print(e)   # show error instead of crashing

# update owner name
acc.owner_name = "Alice Smith"

# save account details to a file
with open('account_summary.txt', 'w') as af:
    af.write(str(acc))   # __str__ makes it look nice

# read file back and show content
with open('account_summary.txt', 'r') as af:
    content = af.read()
    print(f"account_summary.txt: {content}")


Not enough funds.
account_summary.txt: Account(12345, Owner: Alice Smith, Balance: 700)


### Task 2.3 – Extending Behavior
- Demonstrate interest and multiple accounts.

In [19]:
# make a list of bank accounts
accounts = [
    BankAccount("111", "Bob", 1000),
    BankAccount("222", "Carol", 1500),
    BankAccount("333", "Dave", 2000)
]

# go through each account
for acc in accounts:
    acc.apply_interest()   # add 5% interest
    print(acc)             # show account details

Account(111, Owner: Bob, Balance: 1050.0)
Account(222, Owner: Carol, Balance: 1575.0)
Account(333, Owner: Dave, Balance: 2100.0)
